In [1]:
import pandas as pd

# ── Carga ─────────────────────────────────────────────────────────────────────
orders      = pd.read_csv('orders_dataset.csv')
items       = pd.read_csv('order_items_dataset.csv')
products    = pd.read_csv('products_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')
translation.columns = translation.columns.str.replace('\ufeff', '')

# ── Join ──────────────────────────────────────────────────────────────────────
df = (
    items
    .merge(orders[['order_id', 'order_status']], on='order_id')
    .merge(products[['product_id', 'product_category_name']], on='product_id')
    .merge(translation, on='product_category_name', how='left')
)
df['category'] = df['product_category_name_english'].fillna(df['product_category_name'])

# ── Escenarios ────────────────────────────────────────────────────────────────
escenario_a = df
escenario_b = df[~df['order_status'].isin(['canceled', 'unavailable'])]

# ── Resultados ────────────────────────────────────────────────────────────────
for nombre, d in [('Todos los pedidos', escenario_a), ('Excluye cancelados + unavailable', escenario_b)]:
    vol = d.groupby('category')['order_item_id'].count().sort_values(ascending=False).head(5)
    rev = d.groupby('category')['price'].sum().sort_values(ascending=False).head(5)

    print(f"\n{'='*55}")
    print(f" {nombre}")
    print(f"{'='*55}")
    print("\nTOP 5 POR VOLUMEN:")
    for i, (cat, val) in enumerate(vol.items(), 1):
        print(f"  {i}. {cat}: {val:,} unidades")
    print("\nTOP 5 POR INGRESOS:")
    for i, (cat, val) in enumerate(rev.items(), 1):
        print(f"  {i}. {cat}: R$ {val:,.2f}")


 Todos los pedidos

TOP 5 POR VOLUMEN:
  1. bed_bath_table: 11,115 unidades
  2. health_beauty: 9,670 unidades
  3. sports_leisure: 8,641 unidades
  4. furniture_decor: 8,334 unidades
  5. computers_accessories: 7,827 unidades

TOP 5 POR INGRESOS:
  1. health_beauty: R$ 1,258,681.34
  2. watches_gifts: R$ 1,205,005.68
  3. bed_bath_table: R$ 1,036,988.68
  4. sports_leisure: R$ 988,048.97
  5. computers_accessories: R$ 911,954.32

 Excluye cancelados + unavailable

TOP 5 POR VOLUMEN:
  1. bed_bath_table: 11,097 unidades
  2. health_beauty: 9,634 unidades
  3. sports_leisure: 8,590 unidades
  4. furniture_decor: 8,298 unidades
  5. computers_accessories: 7,781 unidades

TOP 5 POR INGRESOS:
  1. health_beauty: R$ 1,255,695.13
  2. watches_gifts: R$ 1,198,185.21
  3. bed_bath_table: R$ 1,035,964.06
  4. sports_leisure: R$ 979,740.92
  5. computers_accessories: R$ 904,322.02
